# Convert a finished gradient checkpoint into NCM-iWSP-GPT1

This is a **new** notebook. Do not open, edit, or overwrite the v10 training kernel `nicholashuihk/ncm-gpt-1-train`, and do not write into its outputs.

Add two Kaggle inputs:

1. This repository (so `kaggle/convert_iwsp.py` and `ncm_iwsp_gpt1` are on disk).
2. The v10 export dataset. Expected file names inside that dataset:
   - `tokenizer.pt`
   - `generator_step100000.pt` (preferred), or `generator.pt`, or `latest.pt`

The converter only reads those inputs. It writes to `/kaggle/working`. Conversion is a closed-form copy plus the tied-head expansion. It is not a 100k training run. CPU is enough. Leave the smoke forward off unless you want one random forward on GPU.


In [ ]:
from pathlib import Path

# Edit this slug to the dataset that holds the v10 export.
INPUT_DIR = Path("/kaggle/input/REPLACE_WITH_DATASET_SLUG")
OUT = Path("/kaggle/working/ncm_iwsp_gpt1_from_100k.pt")

# Optional one-forward check. Not a training loop.
SMOKE = False
DEVICE = "cpu"  # set to "cuda" only when SMOKE is True and a GPU is attached

roots = [Path.cwd()]
kaggle_input = Path("/kaggle/input")
if kaggle_input.is_dir():
    roots.extend(path for path in kaggle_input.iterdir() if path.is_dir())
script = None
for root in roots:
    candidate = root / "kaggle" / "convert_iwsp.py"
    if candidate.is_file():
        script = candidate
        break
if script is None:
    raise SystemExit("Add this repository as a Kaggle input. Do not use the ncm-gpt-1-train kernel.")
print("script", script)
print("tokenizer", INPUT_DIR / "tokenizer.pt")
print("out", OUT)


In [ ]:
import subprocess
import sys

command = [
    sys.executable,
    str(script),
    "--input-dir",
    str(INPUT_DIR),
    "--out",
    str(OUT),
]
if SMOKE:
    command.extend(["--smoke-forward", "--device", DEVICE])
print(" ".join(command))
subprocess.check_call(command)
